# Training and fine tuning

In [ ]:
import optuna
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

### baseline model

In [ ]:
#load csv
df = pd.read_csv('../data/processed/')

In [ ]:
df_base = df.sort_values(by=['resto_name', 'date'])

In [ ]:
# Using shift(1) is absolutely necessary to prevent data leakage 
# (it ensures we only use the *past* 7 days to predict *today*)
df_base['baseline_pred_7d_avg'] = (
    df_base.groupby('resto_name')['total_meals']
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean().shift(1))
)

In [ ]:
split_index = int(len(df_base) * 0.8)
test_df = df_base.iloc[split_index:].copy()

In [ ]:
#Drop rows where the baseline is NaN 
test_clean = test_df.dropna(subset=['baseline_pred_7d_avg', 'total_meals'])

In [ ]:
mae_baseline = mean_absolute_error(test_clean['total_meals'], test_clean['baseline_pred_7d_avg'])
rmse_baseline = np.sqrt(mean_squared_error(test_clean['total_meals'], test_clean['baseline_pred_7d_avg']))

In [ ]:
print("--- 7-Day Moving Average Baseline ---")
print(f"Mean Absolute Error (MAE): {mae_baseline:.2f} meals")
print(f"Root Mean Squared Error (RMSE): {rmse_baseline:.2f} meals")

### Catboost

In [ ]:
target_cols = ['breakfast', 'launch', 'dinner']
X = df.drop(columns=target_cols)
y = df[target_cols]

In [ ]:
split_index = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:
categorical_features = ['resto_name',]

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
base_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MAE', 
    cat_features=categorical_features,
    random_seed=42,
    verbose=False,
    task_type='GPU')

In [ ]:
multi_target_model = MultiOutputRegressor(base_model)

print("Training models via MultiOutputRegressor...")
# Under the hood, this trains 3 completely separate CatBoost models!
multi_target_model.fit(X_train, y_train,
    early_stopping_rounds=50,  
    use_best_model=True)

In [ ]:
preds = multi_target_model.predict(X_test)

mae_b = mean_absolute_error(y_test['breakfast_total'], preds[:, 0])
mae_l = mean_absolute_error(y_test['lunch_total'], preds[:, 1])
mae_d = mean_absolute_error(y_test['dinner_total'], preds[:, 2])

print(f"Breakfast MAE: {mae_b:.2f}")
print(f"Lunch MAE:     {mae_l:.2f}")
print(f"Dinner MAE:    {mae_d:.2f}")